# Somo la 13 - Kumbukumbu ya Wakala na Miundombinu ya Maarifa ya Cognee


## Setup

Daftari hili linaonyesha jinsi ya kujenga msaidizi mahiri wa **kuchora msimbo** mwenye kumbukumbu ya kudumu kwa kutumia grafu za maarifa za [**Cognee**](https://www.cognee.ai/) na **Microsoft Agent Framework** (MAF).

Cognee hubadilisha maandishi yasiyo na muundo kuwa grafu ya maarifa yenye muundo na inaweza kuhojiwa, inayotegemea vector embeddings — ikimpa wakala wako kumbukumbu ndefu yenye uhusiano tajiri.

### Utajifunza Nini
1. **Jenga Grafu za Maarifa**: Badilisha wasifu wa waendelezaji na mbinu bora kuwa maarifa yenye muundo na yanaweza kuhojiwa.
2. **Unganisha Cognee na MAF**: Tumia shughuli za `@tool` kumruhusu wakala wa MAF kuhoji grafu ya maarifa ya Cognee.
3. **Mazungumzo Yenye Ufahamu wa Kikao**: Hifadhi muktadha kati ya maswali mengi katika kikao kimoja.
4. **Kumbukumbu ya Muda Mrefu**: Hifadhi maarifa muhimu kati ya vikao na uyazidishe katika mazungumzo mapya.

### Mahitaji
- Python 3.9+
- Redis inayoendesha ndani ya eneo lako (`docker run -d -p 6379:6379 redis`) kwa usimamizi wa vikao
- Funguo ya API ya LLM (mfano OpenAI) — weka `LLM_API_KEY` katika `.env`
- `CACHING=true` katika `.env` (inahitajika kwa vikao vya Cognee)
- Mradi wa Azure AI Foundry wenye modeli ya mazungumzo iliyotumika
- `AZURE_AI_PROJECT_ENDPOINT` na `AZURE_AI_MODEL_DEPLOYMENT_NAME` katika `.env`
- Azure CLI imeingia kwa ufanisi (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Aina za Kumbukumbu za Wakala

Daftari hili linachunguza aina tatu za kumbukumbu kutoka kwenye daftari kuu la Somo la 13, lakini linatumia Cognee kama mfumo wa kumbukumbu ya muda mrefu:

| Aina ya Kumbukumbu | Mfumo | Muda wa Kuishi |
|---|---|---|
| **Ya Kazi** | `agent.create_session()` (MAF) | Mazungumzo moja |
| **Ya Muda Mfupi** | Hifadhi ya kikao cha Cognee (Redis) | Kikao kimoja |
| **Ya Muda Mrefu** | Mchoro wa maarifa wa Cognee + vekta | Kudumu |

### Msingi wa Kumbukumbu ya Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Andaa Hifadhi ya Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Sehemu ya 1 — Kujenga Hifadhi ya Maarifa

Tunachukua aina tatu za data kuunda hifadhi kamili ya maarifa kwa msaidizi wetu wa uandishi wa programu:

1. **Wasifu wa Mtaalamu** — taaluma binafsi na historia ya kiufundi
2. **Misingi Bora ya Python** — Zen ya Python pamoja na miongozo ya vitendo
3. **Mazungumzo ya Kale** — vipindi vya zamani vya maswali na majibu kati ya watengenezaji na wasaidizi wa AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Onyesha Mchoro wa Maarifa

Cognee inaweza kuunda uonyesho wa HTML unaoingiliana wa vitu na uhusiano ambayo imeviondoa.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Boreshaji Kumbukumbu na Memify

`memify()` huchambua grafu ya maarifa na kuunda sheria za akili — kutambua mifumo, mbinu bora, na mahusiano kati ya dhana.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Sehemu ya 2 — Wakala wa MAF na Zana za Cognee

Sasa tunaunda wakala wa MAF ambaye anaweza kuuliza grafu ya maarifa ya Cognee kupitia kazi za `@tool`. Hii inamruhusu wakala kutumia nguvu kamili ya utafutaji wa semantiki unaojua grafu huku akidumisha muktadha wa mazungumzo kupitia vikao.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Kazi Ya Kumbukumbu Kwa Vikao

`AgentSession` (iliyoanzishwa kupitia `agent.create_session()`) hutoa kumbukumbu ya kazi ndani ya kikao. Mwakilishi anaweza kurejelea ujumbe wa awali huku pia akiuliza grafu ya maarifa ya muda mrefu ya Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Kikao Kipya — Kumbukumbu Zaidi Zinaendelea

Kuanza kikao kipya hufuta kumbukumbu ya kazi, lakini grafu ya maarifa ya Cognee bado ipo. Wakala anaweza kupata maarifa ya muda mrefu yale yale katika mazungumzo mapya kabisa.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Muhtasari

Katika daftari hii umejenga msaidizi wa kufanyia kazi wa msimbo unaounganisha **kumbukumbu ya kazi ya MAF** (`agent.create_session()`) na **mchoro wa maarifa ya muda mrefu wa Cognee**.

### Umejifunza Nini
1. **Ujenzi wa mchoro wa maarifa**: Cognee huchukua maandishi yasiyo na muundo na kujenga mchoro + kumbukumbu ya vector.
2. **Kunaongeza kwa mchoro kwa memify**: Ukweli uliochimbwa na uhusiano tajiri juu ya mchoro wako uliopo.
3. **Muunganiko wa MAF + Cognee**: Funktionen za `@tool` huruhusu wakala wa MAF kuuliza mchoro wa Cognee kwa njia ya asili.
4. **Kumbukumbu ya kazi + kumbukumbu ya muda mrefu**: `AgentSession` (kupitia `agent.create_session()`) hutoa muktadha wa kikao wakati Cognee inatoa maarifa endelevu.
5. **Utafutaji uliosafishwa kwa NodeSets**: Lenga sehemu maalum za mchoro wa maarifa (kwa mfano kanuni pekee).

### Muhimu Kuhitajika
- **Cognee** hubadilisha maandishi ghafi kuwa kumbukumbu yenye muundo na ufahamu wa uhusiano — yenye nguvu zaidi kuliko hifadhi ya vector isiyo na muundo.
- **Funktioni za `@tool`** huzungumza kati ya maajenti wa MAF na mifumo ya maarifa ya nje kwa usafi.
- **`AgentSession`** (kupitia `agent.create_session()`) huhifadhi muktadha wa mazungumzo tofauti na maarifa ya muda mrefu.
- Mchoro wa maarifa ule ule hutumikia vikao vingi na maajenti mbalimbali.

### Matumizi Halisi Ulimwenguni
- **Msaidizi wa waendelezaji**: Ukaguzi wa msimbo, uchambuzi wa matukio, wasaidizi wa usanifu
- **Msaidizi kwa wateja**: Maajenti wa msaada juu ya nyaraka za bidhaa, maswali yanayoulizwa mara kwa mara, na maelezo ya CRM
- **Msaidizi wa wataalamu wa ndani**: Sera, sheria, au wasaidizi wa usalama wanaofanya maamuzi juu ya miongozo
- **Tabaka moja la data**: Changanya data yenye muundo na isiyokuwa na muundo kuwa mchoro mmoja unaoweza kuombwa

### Hatua Zifuatazo
- Jaribu ufahamu wa wakati katika Cognee
- Tafsiri ontolojia ya OWL kwa ubora wa mchoro maalum wa sekta
- Ongeza mizunguko ya maoni ya watumiaji ili kuboresha upatikanaji kwa muda
- Panua kwa mifumo ya maajenti wengi inayoshiriki tabaka moja la kumbukumbu la Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kiarifu cha Hati**:  
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kuwa tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati asilia katika lugha yake ya asili inapaswa kuzingatiwa kama chanzo cha mamlaka. Kwa taarifa muhimu sana, inashauriwa kutumia tafsiri ya mtaalamu binadamu. Hatutawajibika kwa maelezo potofu au uelewa mbaya unaotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
